# Proyecto Final – Text Mining & Image Recognition
## Problema 2: Fruits and Vegetables Recognizer (CNN)

**Objetivo:** clasificar imágenes de al menos 3 frutas y 3 verduras con una red neuronal convolucional robusta a variaciones de tamaño, rotación e iluminación.

**Estructura del notebook**
1. Configuración del entorno
2. Descarga del dataset (Kaggle)
3. Selección de clases y división Train / Validación / Test
4. Exploración del dataset
5. Preprocesamiento y Data Augmentation (`ImageDataGenerator` + `flow_from_directory`)
6. Diseño de 3 arquitecturas CNN
7. Entrenamiento
8. Comparación de arquitecturas
9. Evaluación del mejor modelo en Test
10. Prueba de robustez (rotación, escala, iluminación)
11. Predicción de imágenes nuevas
12. Conclusiones

> En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** antes de ejecutar.

## 1. Configuración del entorno

In [ ]:
import os, random, hashlib, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 2. Obtener el dataset (elige UNA opción)

**No hace falta subir la carpeta descomprimida.** Tienes dos opciones:

**Opción A – `kaggle` (recomendada, la más rápida):** Colab descarga el dataset directo de Kaggle, sin pasar por tu computadora.
- Si pide autenticación: en kaggle.com → *Settings → API → Create New Token*. Eso descarga `kaggle.json`, que subes con la celda opcional.

**Opción B – `drive`:** si ya lo descargaste en tu PC:
1. Sube a tu Google Drive el **archivo .zip** que bajaste de Kaggle, **sin descomprimirlo**. Un solo archivo sube mucho más rápido que 12,000 imágenes sueltas.
2. Escribe su ruta en `ZIP_EN_DRIVE`, por ejemplo `/content/drive/MyDrive/archive.zip`.

Estructura esperada (la misma de tu carpeta):
```
Fruits_Vegetables_Dataset(12000)/
├── Fruits/      FreshApple, FreshBanana, FreshMango, FreshOrange, FreshStrawberry, Rotten...
└── Vegetables/  FreshBellpepper, FreshCarrot, FreshCucumber, FreshPotato, FreshTomato, Rotten...
```

In [ ]:
ORIGEN = "kaggle"                                   # "kaggle" o "drive"
ZIP_EN_DRIVE = "/content/drive/MyDrive/archive.zip" # solo si ORIGEN = "drive"

if ORIGEN == "kaggle":
    # (Opcional) solo si la descarga falla por autenticación:
    # from google.colab import files
    # files.upload()  # sube kaggle.json
    # !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kagglehub
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("muhriddinmuxiddinov/fruits-and-vegetables-dataset")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/dataset"
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    shutil.copy(ZIP_EN_DRIVE, "/content/dataset.zip")      # copiar a disco local = lectura más rápida
    !unzip -q /content/dataset.zip -d /content/dataset

print("Dataset en:", DATA_ROOT)
# Mostrar la estructura de carpetas (hasta 3 niveles) con cantidad de imágenes
for root, dirs, files in sorted(os.walk(DATA_ROOT)):
    nivel = root.replace(DATA_ROOT, "").count(os.sep)
    if nivel <= 3:
        n_img = sum(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files)
        print("   " * nivel + os.path.basename(root) + (f"  ({n_img} imágenes)" if n_img else ""))

## 3. Selección de clases y división Train / Validación / Test

Se eligen **3 frutas y 3 verduras**. Se escogieron pares visualmente parecidos (naranja vs zanahoria por color, manzana vs papa por forma) para que el problema sea exigente y la comparación entre arquitecturas sea significativa.

**Decisión sobre Fresh/Rotten:** la tarea es reconocer el *tipo* de fruta/verdura, no su estado. Por defecto se usan solo imágenes **frescas** (`INCLUIR_PODRIDAS = False`). Si lo cambias a `True`, las podridas se suman a su misma clase (más datos y más variabilidad visual, pero un problema más difícil).

Pasos que hace la celda:
- Busca las carpetas de cada clase ignorando el prefijo `Fresh`/`Rotten`, mayúsculas, espacios y guiones.
- **Elimina duplicados** con hash MD5 (algunos datasets repiten imágenes entre train y test, lo que inflaría la exactitud → *data leakage*).
- Descarta archivos corruptos y convierte todo a RGB/JPG.
- Divide de forma **estratificada 70% / 15% / 15%**.

In [ ]:
FRUTAS   = ["apple", "banana", "orange"]
VERDURAS = ["carrot", "potato", "cucumber"]
CLASES = FRUTAS + VERDURAS

INCLUIR_PODRIDAS = False
MAX_POR_CLASE = 1500          # tope para balancear / acelerar
WORK_DIR = "/content/data"
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def parsear_carpeta(nombre):
    # 'FreshApple' -> ('apple', 'fresh');  'Rotten_Carrot' -> ('carrot', 'rotten')
    n = nombre.lower().replace(" ", "").replace("_", "").replace("-", "")
    for estado in ("fresh", "rotten"):
        if n.startswith(estado):
            return n[len(estado):], estado
    return n, "fresh"

def recolectar(clase):
    rutas, hashes = [], set()
    for root, dirs, files in os.walk(DATA_ROOT):
        base, estado = parsear_carpeta(os.path.basename(root))
        if estado == "rotten" and not INCLUIR_PODRIDAS:
            continue
        if base == clase or base == clase + "s":
            for f in files:
                if f.lower().endswith(EXT):
                    p = os.path.join(root, f)
                    h = hashlib.md5(open(p, "rb").read()).hexdigest()
                    if h not in hashes:
                        hashes.add(h); rutas.append(p)
    random.shuffle(rutas)
    return rutas[:MAX_POR_CLASE]

def copiar(rutas, destino):
    os.makedirs(destino, exist_ok=True)
    ok = 0
    for i, p in enumerate(rutas):
        try:
            img = Image.open(p).convert("RGB")
            img.save(os.path.join(destino, f"{i:05d}.jpg"), quality=95)
            ok += 1
        except Exception:
            pass  # imagen corrupta
    return ok

shutil.rmtree(WORK_DIR, ignore_errors=True)
resumen = []
for c in CLASES:
    rutas = recolectar(c)
    assert len(rutas) > 0, f"No se encontraron imágenes para '{c}'. Revisa los nombres impresos en la celda anterior."
    tr, tmp = train_test_split(rutas, test_size=0.30, random_state=SEED)
    va, te  = train_test_split(tmp,   test_size=0.50, random_state=SEED)
    n = {s: copiar(r, f"{WORK_DIR}/{s}/{c}") for s, r in [("train", tr), ("val", va), ("test", te)]}
    resumen.append({"clase": c, "tipo": "fruta" if c in FRUTAS else "verdura", **n})

df_resumen = pd.DataFrame(resumen)
df_resumen["total"] = df_resumen[["train", "val", "test"]].sum(axis=1)
df_resumen

## 4. Exploración del dataset

In [ ]:
df_resumen.set_index("clase")[["train", "val", "test"]].plot(kind="bar", stacked=True, figsize=(8,4))
plt.title("Imágenes por clase y conjunto"); plt.ylabel("Cantidad"); plt.xticks(rotation=0); plt.show()

fig, axes = plt.subplots(len(CLASES), 5, figsize=(12, 2.4*len(CLASES)))
for i, c in enumerate(CLASES):
    carpeta = f"{WORK_DIR}/train/{c}"
    muestras = random.sample(os.listdir(carpeta), 5)
    for j, f in enumerate(muestras):
        img = Image.open(os.path.join(carpeta, f))
        axes[i, j].imshow(img); axes[i, j].axis("off")
        if j == 0: axes[i, j].set_title(f"{c}  {img.size}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

**Observación:** las imágenes tienen tamaños, fondos, ángulos e iluminación muy distintos. Esto justifica (a) redimensionar a un tamaño estándar y (b) aplicar data augmentation.

## 5. Preprocesamiento y Data Augmentation

### Decisión: ¿escala de grises o color?
Se trabaja **a color (RGB)**. El color es una de las características más discriminantes en este problema: naranja vs zanahoria vs pepino se diferencian principalmente por el tono, y una banana madura es casi imposible de confundir por su amarillo. En escala de grises la red tendría que depender solo de forma y textura, perdiendo información valiosa. El costo extra (3 canales en lugar de 1) solo afecta la primera capa convolucional.

### Parámetros
- **Tamaño estándar:** 128×128 px (balance entre detalle y velocidad).
- **Normalización:** `rescale=1./255` → valores entre 0 y 1.
- **Augmentation (solo en entrenamiento):** rotación, escalado (zoom), desplazamiento, flip horizontal, brillo y **ruido gaussiano** (función propia vía `preprocessing_function`).
- Validación y Test **solo se normalizan**, para medir el desempeño sobre imágenes reales sin alterar.

In [ ]:
IMG_SIZE = (128, 128)
BATCH = 32
NUM_CLASES = len(CLASES)

def ruido_gaussiano(img):
    # Se aplica antes del rescale, por eso trabaja en escala 0-255
    if np.random.rand() < 0.5:
        img = img + np.random.normal(0, 12.0, img.shape)
    return np.clip(img, 0, 255)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,          # rotación
    zoom_range=0.25,            # escalado
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    horizontal_flip=True,       # flip horizontal
    brightness_range=[0.6, 1.4],# iluminación
    fill_mode="nearest",
    preprocessing_function=ruido_gaussiano,  # ruido
)
eval_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(f"{WORK_DIR}/train", target_size=IMG_SIZE, color_mode="rgb",
                                              class_mode="categorical", batch_size=BATCH, shuffle=True, seed=SEED)
val_gen   = eval_datagen.flow_from_directory(f"{WORK_DIR}/val", target_size=IMG_SIZE, color_mode="rgb",
                                             class_mode="categorical", batch_size=BATCH, shuffle=False)
test_gen  = eval_datagen.flow_from_directory(f"{WORK_DIR}/test", target_size=IMG_SIZE, color_mode="rgb",
                                             class_mode="categorical", batch_size=BATCH, shuffle=False)

IDX2CLASE = {v: k for k, v in train_gen.class_indices.items()}
print(train_gen.class_indices)

In [ ]:
# Visualizar el efecto del augmentation sobre una misma imagen
ejemplo = os.path.join(f"{WORK_DIR}/train/{CLASES[0]}", os.listdir(f"{WORK_DIR}/train/{CLASES[0]}")[0])
x = img_to_array(load_img(ejemplo, target_size=IMG_SIZE))[None, ...]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
axes[0].imshow(x[0].astype("uint8")); axes[0].set_title("original"); axes[0].axis("off")
for i, batch in enumerate(train_datagen.flow(x, batch_size=1, seed=SEED)):
    if i == 7: break
    axes[i+1].imshow(batch[0]); axes[i+1].set_title(f"aug {i+1}"); axes[i+1].axis("off")
plt.show()

## 6. Diseño de las 3 arquitecturas CNN

Cada arquitectura varía deliberadamente los elementos que pide la hoja de trabajo (capas, tipo de pooling, tamaño de kernel, activación, neuronas, optimizador):

| | **A – Baseline** | **B – Profunda + BatchNorm** | **C – Kernels grandes + AvgPool** |
|---|---|---|---|
| Bloques conv | 3 (1 conv c/u) | 4 (2 conv c/u) | 3 (1 conv c/u) |
| Filtros | 32-64-128 | 32-64-128-256 | 32-64-128 |
| Kernel | 3×3 | 3×3 | 5×5, 5×5, 3×3 |
| Pooling | MaxPooling 2×2 | MaxPooling 2×2 + GlobalAveragePooling | AveragePooling 2×2 |
| Activación | ReLU | ReLU + BatchNormalization | ELU |
| Clasificador | Flatten → Dense 128 | GAP → Dense 256 | Flatten → Dense 256 |
| Regularización | Dropout 0.5 | Dropout 0.25/0.5 + L2 | Dropout 0.4 |
| Optimizador | Adam (1e-3) | Adam (1e-3) | SGD + momentum 0.9 (1e-2) |
| Salida | Softmax (6 clases) | Softmax (6 clases) | Softmax (6 clases) |

In [ ]:
INPUT_SHAPE = IMG_SIZE + (3,)

def arquitectura_A():
    m = models.Sequential([
        layers.Input(INPUT_SHAPE),
        layers.Conv2D(32, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASES, activation="softmax"),
    ], name="A_Baseline")
    m.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def bloque_B(x, filtros):
    for _ in range(2):
        x = layers.Conv2D(filtros, 3, padding="same", use_bias=False,
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(0.25)(x)

def arquitectura_B():
    inp = layers.Input(INPUT_SHAPE)
    x = inp
    for f in [32, 64, 128, 256]:
        x = bloque_B(x, f)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASES, activation="softmax")(x)
    m = models.Model(inp, out, name="B_Profunda_BN")
    m.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def arquitectura_C():
    m = models.Sequential([
        layers.Input(INPUT_SHAPE),
        layers.Conv2D(32, 5, activation="elu", padding="same"), layers.AveragePooling2D(),
        layers.Conv2D(64, 5, activation="elu", padding="same"), layers.AveragePooling2D(),
        layers.Conv2D(128, 3, activation="elu", padding="same"), layers.AveragePooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="elu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASES, activation="softmax"),
    ], name="C_Kernel5_AvgPool")
    m.compile(optimizer=optimizers.SGD(1e-2, momentum=0.9), loss="categorical_crossentropy", metrics=["accuracy"])
    return m

ARQUITECTURAS = {"A": arquitectura_A, "B": arquitectura_B, "C": arquitectura_C}
for k, f in ARQUITECTURAS.items():
    m = f(); print(f"{m.name}: {m.count_params():,} parámetros")
arquitectura_B().summary()

## 7. Entrenamiento

Callbacks utilizados:
- **EarlyStopping** (paciencia 8, restaura los mejores pesos) → evita sobreajuste.
- **ReduceLROnPlateau** → baja el learning rate cuando la validación se estanca.
- **ModelCheckpoint** → guarda el mejor modelo según `val_accuracy`.

Se usan **pesos de clase** por si las clases quedaron desbalanceadas.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
pesos = compute_class_weight("balanced", classes=np.unique(train_gen.classes), y=train_gen.classes)
CLASS_WEIGHT = dict(enumerate(pesos))

EPOCHS = 40
os.makedirs("/content/modelos", exist_ok=True)
resultados, historias, modelos_entrenados = [], {}, {}

for clave, constructor in ARQUITECTURAS.items():
    tf.keras.backend.clear_session()
    modelo = constructor()
    cbs = [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        callbacks.ModelCheckpoint(f"/content/modelos/{modelo.name}.keras", monitor="val_accuracy", save_best_only=True),
    ]
    print(f"\n===== Entrenando {modelo.name} =====")
    t0 = time.time()
    h = modelo.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                   class_weight=CLASS_WEIGHT, callbacks=cbs, verbose=1)
    dur = time.time() - t0

    val_loss, val_acc = modelo.evaluate(val_gen, verbose=0)
    test_loss, test_acc = modelo.evaluate(test_gen, verbose=0)
    historias[clave], modelos_entrenados[clave] = h.history, modelo
    resultados.append({"arquitectura": modelo.name, "parámetros": modelo.count_params(),
                       "épocas": len(h.history["loss"]), "tiempo (min)": round(dur/60, 1),
                       "train_acc": round(max(h.history["accuracy"]), 4),
                       "val_acc": round(val_acc, 4), "test_acc": round(test_acc, 4),
                       "test_loss": round(test_loss, 4)})

## 8. Comparación de arquitecturas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for clave, h in historias.items():
    nombre = modelos_entrenados[clave].name
    axes[0].plot(h["val_accuracy"], label=f"{nombre} val")
    axes[0].plot(h["accuracy"], "--", alpha=0.5, label=f"{nombre} train")
    axes[1].plot(h["val_loss"], label=f"{nombre} val")
    axes[1].plot(h["loss"], "--", alpha=0.5, label=f"{nombre} train")
axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
for a in axes: a.set_xlabel("Época"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.show()

df_res = pd.DataFrame(resultados).sort_values("val_acc", ascending=False).reset_index(drop=True)
df_res

In [ ]:
# El mejor modelo se elige por VALIDACIÓN (el test se reserva para la evaluación final)
mejor_nombre = df_res.loc[0, "arquitectura"]
mejor_clave = [k for k, m in modelos_entrenados.items() if m.name == mejor_nombre][0]
mejor = modelos_entrenados[mejor_clave]
print("Mejor arquitectura:", mejor_nombre)

## 9. Evaluación del mejor modelo en el conjunto de Test

In [ ]:
test_gen.reset()
probs = mejor.predict(test_gen, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = test_gen.classes
nombres = [IDX2CLASE[i] for i in range(NUM_CLASES)]

print(classification_report(y_true, y_pred, target_names=nombres, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=nombres, yticklabels=nombres)
plt.title(f"Matriz de confusión – {mejor.name}"); plt.xlabel("Predicción"); plt.ylabel("Real"); plt.show()

In [ ]:
# Ejemplos de errores para análisis
errores = np.where(y_pred != y_true)[0]
print(f"Errores: {len(errores)} de {len(y_true)}")
fig, axes = plt.subplots(1, min(6, len(errores)) or 1, figsize=(15, 3))
axes = np.atleast_1d(axes)
for ax, idx in zip(axes, errores[:6]):
    ax.imshow(load_img(test_gen.filepaths[idx], target_size=IMG_SIZE))
    ax.set_title(f"Real: {IDX2CLASE[y_true[idx]]}\nPred: {IDX2CLASE[y_pred[idx]]}", fontsize=9); ax.axis("off")
plt.show()

## 10. Prueba de robustez (tamaño, rotación e iluminación)

El enunciado pide que el modelo sea **robusto** ante estas variaciones. Para demostrarlo, se evalúa el mejor modelo sobre el conjunto de test con transformaciones controladas que nunca vio así.

In [ ]:
escenarios = {
    "Original":            dict(),
    "Rotación ±45°":       dict(rotation_range=45),
    "Escala (zoom ±30%)":  dict(zoom_range=0.3),
    "Oscura (40-60%)":     dict(brightness_range=[0.4, 0.6]),
    "Brillante (140-160%)":dict(brightness_range=[1.4, 1.6]),
    "Flip horizontal":     dict(horizontal_flip=True),
}
filas = []
for nombre_esc, params in escenarios.items():
    gen = ImageDataGenerator(rescale=1./255, **params).flow_from_directory(
        f"{WORK_DIR}/test", target_size=IMG_SIZE, class_mode="categorical",
        batch_size=BATCH, shuffle=False, seed=SEED)
    for clave, m in modelos_entrenados.items():
        _, acc = m.evaluate(gen, verbose=0)
        filas.append({"escenario": nombre_esc, "modelo": m.name, "accuracy": round(acc, 4)})

df_rob = pd.DataFrame(filas).pivot(index="escenario", columns="modelo", values="accuracy").loc[list(escenarios)]
df_rob.plot(kind="bar", figsize=(10, 4)); plt.ylim(0, 1); plt.xticks(rotation=20)
plt.title("Robustez en Test ante transformaciones"); plt.ylabel("Accuracy"); plt.show()
df_rob

## 11. Predicción de imágenes nuevas

In [ ]:
def predecir(ruta, modelo=mejor):
    img = load_img(ruta, target_size=IMG_SIZE)
    x = img_to_array(img)[None, ...] / 255.0
    p = modelo.predict(x, verbose=0)[0]
    top = p.argsort()[::-1][:3]
    plt.imshow(img); plt.axis("off")
    plt.title(" | ".join(f"{IDX2CLASE[i]} {p[i]:.1%}" for i in top)); plt.show()
    return IDX2CLASE[top[0]]

# Prueba con una imagen de test al azar
predecir(random.choice(test_gen.filepaths))

# Para subir tu propia foto:
# from google.colab import files
# for nombre in files.upload(): predecir(nombre)

In [ ]:
# Guardar el mejor modelo (súbelo a GitHub si pesa < 100 MB, o usa Git LFS / Google Drive)
mejor.save("/content/modelos/mejor_modelo.keras")
df_res.to_csv("/content/modelos/comparacion_arquitecturas.csv", index=False)

## 12. Conclusiones

### ¿Por qué la CNN seleccionada funciona mejor que las otras?

> ✏️ **Completa con tus resultados reales** (tabla de la sección 8 y gráfica de robustez). Normalmente la **arquitectura B** gana; si ese es tu caso, estos son los argumentos técnicos:

1. **Mayor profundidad con kernels 3×3 apilados.** Dos convoluciones 3×3 seguidas cubren el mismo campo receptivo que una 5×5 pero con menos parámetros y una no linealidad extra, lo que permite aprender jerarquías más ricas: bordes → texturas (cáscara de naranja, piel de papa) → formas completas.
2. **Batch Normalization.** Estabiliza la distribución de activaciones entre capas; esto acelera la convergencia, permite un learning rate mayor y actúa como regularizador. Además ayuda con los cambios de iluminación, porque normaliza la escala de las activaciones.
3. **Global Average Pooling en lugar de Flatten.** Reduce drásticamente los parámetros de la parte densa (en A y C el `Flatten` concentra la mayoría de los parámetros), disminuye el sobreajuste y hace al modelo menos dependiente de la **posición** exacta del objeto → más robusto a escala y desplazamiento.
4. **Regularización combinada** (Dropout + L2 + augmentation): la brecha entre `train_acc` y `val_acc` es menor que en A.
5. **Frente a C:** el AveragePooling suaviza la señal y "diluye" rasgos distintivos locales, mientras que MaxPooling conserva la activación más fuerte (el rasgo más relevante). Los kernels 5×5 aumentan parámetros sin aportar más capacidad jerárquica, y SGD converge más lento en el mismo número de épocas.
6. **Frente a A:** la baseline es poco profunda; aprende color y forma general, pero confunde clases de color/forma similar (revisar en la matriz de confusión pares como naranja–zanahoria o manzana–papa).

### Decisiones de diseño resumidas
- **Color (RGB)** porque el color es altamente discriminante entre frutas y verduras.
- **128×128** como tamaño estándar; **normalización 0-1** con `rescale=1./255`.
- **Augmentation** (rotación, zoom, desplazamiento, flip, brillo, ruido) solo en entrenamiento, validado con la prueba de robustez.
- División **70/15/15 estratificada** y **sin duplicados** entre conjuntos para evitar fuga de información.